### Notebook : 06_vector_search_agent


##### 1. Notebook Purpose

The Vector Search Agent retrieves semantically similar customer notes when requested by the Coordinator Agent.

It:

- Reads the Coordinator execution plan.
- Finds the task assigned to vector_search_agent.
- Validates task dependencies.
- Uses the task description or user request as the semantic-search query.
- Calls the injected Vector Search Tool.
- Validates and normalizes the tool response.
- Creates a validated VectorSearchAgentResult.
- Stores the result in shared state.
- Records execution history and errors.
- Skips cleanly when no Vector Search task is assigned.

The Vector Search Agent does not decide whether semantic search is needed. That decision belongs to the Coordinator Agent.


##### 2. Technologies Used

- Python
- Databricks Vector Search
- Embedding models
- Semantic search
- Pydantic
- TypedDict
- Dependency injection
- Shared multi-agent state
- Unit testing with mock functions
- Integration test with VectorSearch


##### 3. Input

- state: MultiAgentState
- vector_search_tool: VectorSearchToolFunction
- The shared state should contain:
    - state["user_request"]
    - state["coordinator_result"]
    - state["agent_results"]
    - state["execution_history"]
    - state["errors"]


###### 4. Output

- Updated MultiAgentState containing state["agent_results"]["vector_search_agent"]


##### 5. Architecture

```text

Coordinator Agent
        │
        ▼
CoordinatorResult
        │
        ▼
Vector Search Agent
        │
        ├── Find assigned task
        ├── Validate task dependencies
        ├── Resolve semantic-search query
        ├── Call Vector Search Tool
        ├── Validate tool response
        ├── Create VectorSearchAgentResult
        └── Store result in shared state
        │
        ▼
Updated MultiAgentState

```

##### 6 : Install Vector Search client

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()


##### 7. Load Shared Models and Helpers

In [0]:
%run ./00A_create_vector_search_resources

In [0]:
%run ./01_shared_models

In [0]:
%run ./02_shared_state_and_helpers


##### 8. Imports

In [0]:
from typing import Any, Callable, Dict, List, Optional

from pydantic import ValidationError
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

##### 9. Configuration Constants

In [0]:
# Unity Catalog
CATALOG = "dbw_agentic_ai_dev"
SCHEMA = "telco_ai"

# Tables
EMBEDDINGS_TABLE = (
    f"{CATALOG}.{SCHEMA}.customer_note_embeddings"
)

# Foundation Model
EMBEDDING_MODEL = "databricks-gte-large-en"

# Vector Search
VECTOR_SEARCH_ENDPOINT_NAME = (
    "telco-vector-search-endpoint"
)

VECTOR_INDEX_NAME = (
    f"{CATALOG}.{SCHEMA}.customer_note_embeddings_index"
)

DEFAULT_NUM_RESULTS = 3

##### 10. Connect to Vector Search

In [0]:
from databricks.vector_search.client import (
    VectorSearchClient,
)

vs_client = VectorSearchClient()

index = vs_client.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=VECTOR_INDEX_NAME,
)


##### 11. Resolve the Semantic-Search Query

In [0]:
def resolve_search_query(
    task: AgentTask,
    user_request: str,
) -> str:
    """
    Determine the query that should be sent to the
    Vector Search Tool.

    The task description is preferred. The original user request
    is used when the task description is empty.
    """

    task_description = task.task_description.strip()

    if task_description:
        return task_description

    normalized_user_request = user_request.strip()

    if normalized_user_request:
        return normalized_user_request

    raise ValueError(
        "Vector Search Agent could not determine a semantic-search query."
    )


##### 12. Convert Tool Values to Python Types

In [0]:
def convert_vector_search_value(
    value: Any,
) -> Any:
    """
    Convert Vector Search Tool values into standard Python objects.
    """
    print(value)
    
    if value is None:
        return None

    if hasattr(value, "asDict"):
        return {
            key: convert_vector_search_value(item)
            for key, item in value.asDict(
                recursive=True
            ).items()
        }

    if isinstance(value, dict):
        return {
            key: convert_vector_search_value(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            convert_vector_search_value(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            convert_vector_search_value(item)
            for item in value
        ]

    if hasattr(value, "item"):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass

    return value


##### 13. Normalize Similarity Score

In [0]:
def normalize_similarity_score(
    score: Any,
) -> Optional[float]:
    """
    Normalize a similarity score to a float between 0.0 and 1.0.

    A missing score is allowed because some tool implementations
    may not expose it.
    """

    if score is None:
        return None

    score = convert_vector_search_value(score)

    try:
        normalized_score = float(score)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid similarity score: {score!r}"
        ) from exc

    if not 0.0 <= normalized_score <= 1.0:
        raise ValueError(
            "Similarity score must be between 0.0 and 1.0."
        )

    return normalized_score


##### 14. Validate One Search Result

In [0]:
def validate_search_result(
    search_result: Dict[str, Any],
    result_position: int,
) -> Dict[str, Any]:
    """
    Validate and normalize one semantic-search result.
    """

    if not isinstance(search_result, dict):
        raise TypeError(
            f"Search result {result_position} must be a dictionary."
        )

    normalized_result = convert_vector_search_value(
        search_result
    )

    customer_id = normalized_result.get(
        "customer_id"
    )

    note = normalized_result.get(
        "note"
    )

    similarity_score = normalized_result.get(
        "similarity_score",
        normalized_result.get("score"),
    )

    if customer_id is None:
        raise ValueError(
            f"Search result {result_position} is missing "
            "'customer_id'."
        )

    if note is None:
        raise ValueError(
            f"Search result {result_position} is missing 'note'."
        )

    customer_id = str(customer_id).strip()
    note = str(note).strip()

    if not customer_id:
        raise ValueError(
            f"Search result {result_position} contains an empty "
            "customer ID."
        )

    if not note:
        raise ValueError(
            f"Search result {result_position} contains an empty note."
        )

    normalized_score = normalize_similarity_score(
        similarity_score
    )

    return {
        "customer_id": customer_id,
        "note": note,
        "similarity_score": normalized_score,
    }

##### 15 : Define Similar Customer Notes Tool

In [0]:
def similar_customer_notes_tool(
    question: str,
    num_results: int = 3
):
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    # Generate an embedding for the user question
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    if (
        not response.data
        or response.data[0].embedding is None
    ):
        raise ValueError(
            "The embedding model returned no embedding."
        )

    question_embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    # Search the existing Vector Search index
    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = (
        results
        .get("result", {})
        .get("data_array", [])
    )

    if not rows:
        return {
            "tool": "similar_customer_notes_tool",
            "status": "no_results",
            "context": "",
            "rows": [],
            "result_count": 0
        }

    context_lines = [
        f"Customer {str(customer_id)}: {note}"
        for customer_id, note, score in rows
    ]

    context = "\n".join(context_lines)

    return {
        "tool": "similar_customer_notes_tool",
        "status": "success",
        "context": context,
        "rows": rows,
        "result_count": len(rows)
    }

##### 16. Vector Search Tool

In [0]:
def vector_search_tool(
    query: str,
    num_results: int = 3,
) -> Dict[str, Any]:
    """
    Run semantic search over customer notes and return
    the contract expected by the Vector Search Agent.
    """

    tool_response = similar_customer_notes_tool(
        question=query,
        num_results=num_results,
    )

    status = tool_response.get("status")

    if status == "no_results":
        return {
            "tool": "vector_search_tool",
            "status": "success",
            "query": query,
            "results": [],
        }

    if status != "success":
        raise RuntimeError(
            "Similar Customer Notes Tool failed: "
            f"{tool_response}"
        )

    rows = tool_response.get(
        "rows",
        [],
    )

    results = []

    for row in rows:
        customer_id, note, score = row

        results.append(
            {
                "customer_id": str(customer_id),
                "note": str(note),
                "similarity_score": float(score),
            }
        )

    return {
        "tool": "vector_search_tool",
        "status": "success",
        "query": query,
        "results": results,
    }


##### 17. Validate Vector Search Tool Response

In [0]:
def validate_vector_search_tool_response(
    tool_response: Dict[str, Any],
    expected_query: str,
) -> Dict[str, Any]:
    """
    Validate and normalize the Vector Search Tool response.

    Parameters
    ----------
    tool_response:
        Raw dictionary returned by the Vector Search Tool.

    expected_query:
        Query originally submitted by the Vector Search Agent.

    Returns
    -------
    Dict[str, Any]
        Normalized query and search results.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "Vector Search Tool must return a dictionary."
        )

    normalized_response = convert_vector_search_value(
        tool_response
    )

    status = normalized_response.get("status")

    if status != "success":
        error_message = normalized_response.get(
            "message",
            "Vector Search Tool execution failed.",
        )

        raise RuntimeError(error_message)

    returned_query = normalized_response.get(
        "query",
        expected_query,
    )

    returned_query = str(returned_query).strip()

    if not returned_query:
        returned_query = expected_query

    results = normalized_response.get("results")

    if results is None:
        raise ValueError(
            "Vector Search Tool response is missing 'results'."
        )

    if not isinstance(results, list):
        raise TypeError(
            "Vector Search Tool 'results' must be a list."
        )

    validated_results = [
        validate_search_result(
            search_result=result,
            result_position=index,
        )
        for index, result in enumerate(
            results,
            start=1,
        )
    ]

    return {
        "query": returned_query,
        "results": validated_results,
        "result_count": len(validated_results),
        "raw_tool_response": normalized_response,
    }


##### 18. Execute the Vector Search Agent

In [0]:
def execute_vector_search_agent(
    user_request: str,
    task: AgentTask,
    vector_search_tool: VectorSearchToolFunction,
    num_results: int = DEFAULT_NUM_RESULTS,
) -> VectorSearchAgentResult:
    """
    Execute the Vector Search Agent's assigned task.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "Vector Search Agent requires a non-empty "
            "customer request."
        )

    if task.agent_name != VECTOR_SEARCH_AGENT_NAME:
        raise ValueError(
            "The assigned task does not belong to "
            "vector_search_agent."
        )

    if num_results <= 0:
        raise ValueError(
            "num_results must be greater than zero."
        )

    query = resolve_search_query(
        task=task,
        user_request=cleaned_request,
    )

    tool_response = vector_search_tool(
        query,
        num_results,
    )

    validated_response = (
        validate_vector_search_tool_response(
            tool_response=tool_response,
            expected_query=query,
        )
    )

    results = validated_response["results"]
    result_count = len(results)

    if result_count == 0:
        message = (
            "Vector Search Agent completed successfully, "
            "but no similar customer notes were found."
        )
    else:
        message = (
            f"Vector Search Agent retrieved "
            f"{result_count} similar customer "
            f"{'note' if result_count == 1 else 'notes'}."
        )

    result_payload = {
        "agent_name": VECTOR_SEARCH_AGENT_NAME,
        "status": "success",
        "message": message,
        "task_description": task.task_description,
        "error": None,
        "task_id": task.task_id,
        "query": validated_response["query"],
        "results": results,
    }

    return VectorSearchAgentResult.model_validate(
        result_payload
    )

##### 19. Run the Vector Search Agent

In [0]:
def run_vector_search_agent(
    state: MultiAgentState,
    vector_search_tool: VectorSearchToolFunction,
    num_results: int = DEFAULT_NUM_RESULTS,
) -> MultiAgentState:
    """
    Run the Vector Search Agent inside the shared workflow.
    """

    try:
        coordinator_result = state.get(
            "coordinator_result"
        )

        if coordinator_result is None:
            raise ValueError(
                "Vector Search Agent cannot run because "
                "coordinator_result is missing."
            )

        assigned_task = find_assigned_agent_task(
            coordinator_result=coordinator_result,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
        )

        if assigned_task is None:
            record_agent_execution(
                state=state,
                agent_name=VECTOR_SEARCH_AGENT_NAME,
                status="skipped",
                message=(
                    "Vector Search Agent was not required "
                    "by the execution plan."
                ),
            )

            return state

        validate_task_dependencies(
            state=state,
            task=assigned_task,
        )

        vector_search_result = (
            execute_vector_search_agent(
                user_request=state["user_request"],
                task=assigned_task,
                vector_search_tool=vector_search_tool,
                num_results=num_results,
            )
        )

        store_agent_result(
            state=state,
            agent_result=vector_search_result,
        )

        record_agent_execution(
            state=state,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            status="success",
            message=(
                "Vector Search task completed successfully."
            ),
        )

    except (
        ValueError,
        TypeError,
        KeyError,
        RuntimeError,
        ValidationError,
    ) as exc:

        error_message = str(exc)

        record_agent_execution(
            state=state,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            status="failed",
            message=(
                "Vector Search Agent failed to complete "
                "the assigned task."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            error_code="VECTOR_SEARCH_AGENT_ERROR",
            error_message=error_message,
        )

    except Exception as exc:

        error_message = (
            "Unexpected Vector Search Agent error: "
            f"{exc}"
        )

        record_agent_execution(
            state=state,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            status="failed",
            message=(
                "Vector Search Agent encountered an "
                "unexpected error."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            error_code=(
                "VECTOR_SEARCH_AGENT_UNEXPECTED_ERROR"
            ),
            error_message=error_message,
        )

    return state

##### 20. Test the Vector Search Agent

In [0]:
test_vector_search_agent_unit_test()

In [0]:
def test_vector_search_agent_unit_test() -> None:
    """
    Run deterministic unit tests for the
    Vector Search Agent.

    Tests:
    1. Successful semantic search.
    2. Successful search with no results.
    3. Vector Search Agent skip behavior.
    4. Vector Search Tool failure.
    5. Missing results field.
    6. Invalid individual search result.
    """

    # =========================================================
    # Shared test helper
    # =========================================================

    def create_mock_vector_search_coordinator_result(
        task_description: str,
    ) -> CoordinatorResult:
        """
        Create a Coordinator result containing one
        Vector Search task.
        """

        return CoordinatorResult(
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            request_type="vector_search",
            reasoning=(
                "The user is asking for information "
                "that should be retrieved from customer "
                "notes using semantic search."
            ),
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=(
                        VECTOR_SEARCH_AGENT_NAME
                    ),
                    task_description=(
                        task_description
                    ),
                    depends_on=[],
                )
            ],
        )

    # =========================================================
    # Shared mock tools
    # =========================================================

    def mock_successful_vector_search_tool(
        query: str,
        num_results: int,
    ) -> Dict[str, Any]:
        """
        Return predictable semantic-search results.
        """

        all_results = [
            {
                "customer_id": "1001",
                "note": (
                    "Customer wants to cancel because "
                    "the monthly charges are too high."
                ),
                "similarity_score": 0.94,
            },
            {
                "customer_id": "1005",
                "note": (
                    "Customer requested service "
                    "termination after repeated "
                    "connectivity problems."
                ),
                "similarity_score": 0.91,
            },
            {
                "customer_id": "1008",
                "note": (
                    "Customer is unhappy with service "
                    "quality and is considering "
                    "cancellation."
                ),
                "similarity_score": 0.88,
            },
        ]

        return {
            "tool": "vector_search_tool",
            "status": "success",
            "query": query,
            "results": (
                all_results[:num_results]
            ),
        }

    def mock_empty_vector_search_tool(
        query: str,
        num_results: int,
    ) -> Dict[str, Any]:
        """
        Return a successful response with no
        matching notes.
        """

        return {
            "tool": "vector_search_tool",
            "status": "success",
            "query": query,
            "results": [],
        }

    def mock_failed_vector_search_tool(
        query: str,
        num_results: int,
    ) -> Dict[str, Any]:
        """
        Simulate a Vector Search Tool failure.
        """

        return {
            "tool": "vector_search_tool",
            "status": "error",
            "query": query,
            "message": (
                "The Vector Search endpoint was "
                "unavailable."
            ),
        }

    def mock_invalid_vector_search_tool(
        query: str,
        num_results: int,
    ) -> Dict[str, Any]:
        """
        Simulate a malformed Vector Search response.
        """

        return {
            "tool": "vector_search_tool",
            "status": "success",
            "query": query,
            # "results" intentionally omitted.
        }

    def mock_invalid_search_result_tool(
        query: str,
        num_results: int,
    ) -> Dict[str, Any]:
        """
        Simulate a response containing an invalid
        individual result.
        """

        return {
            "tool": "vector_search_tool",
            "status": "success",
            "query": query,
            "results": [
                {
                    "customer_id": "1001",
                    # "note" intentionally omitted.
                    "similarity_score": 0.94,
                }
            ],
        }

    # =========================================================
    # TEST 1: Successful semantic search
    # =========================================================

    print("=" * 80)
    print("TEST 1: Successful semantic search")
    print("=" * 80)

    state = create_initial_state(
        "Why are customers likely to cancel service?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes explaining why "
                "customers are likely to cancel service."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_successful_vector_search_tool
        ),
        num_results=3,
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert result.status == "success"
    assert len(result.results) == 3
    assert len(updated_state["errors"]) == 0

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 2: Successful search with no results
    # =========================================================

    print("=" * 80)
    print("TEST 2: Successful search with no results")
    print("=" * 80)

    state = create_initial_state(
        "Find notes about an unknown service issue."
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes about an unknown "
                "service issue."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_empty_vector_search_tool
        ),
    )

    result = updated_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert result.status == "success"
    assert len(result.results) == 0
    assert len(updated_state["errors"]) == 0

    assert (
        "no similar"
        in result.message.lower()
    )

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 3: Vector Search Agent skips
    # =========================================================

    print("=" * 80)
    print("TEST 3: Vector Search Agent skips")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        CoordinatorResult(
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            request_type="sql_analytics",
            reasoning=(
                "The user is requesting a structured "
                "SQL calculation."
            ),
            execution_plan=[
                AgentTask(
                    task_id="task_1",
                    agent_name=SQL_AGENT_NAME,
                    task_description=(
                        "Count the number of churned "
                        "customers."
                    ),
                    depends_on=[],
                )
            ],
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_successful_vector_search_tool
        ),
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 0

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "skipped"
    )

    print("PASS")
    print(
        updated_state[
            "execution_history"
        ][-1]
    )
    print()

    # =========================================================
    # TEST 4: Vector Search Tool failure
    # =========================================================

    print("=" * 80)
    print("TEST 4: Vector Search Tool failure")
    print("=" * 80)

    state = create_initial_state(
        "Why are customers unhappy?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find customer notes explaining "
                "customer dissatisfaction."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_failed_vector_search_tool
        ),
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    print("PASS")
    print(
        updated_state["errors"][-1]
    )
    print()

    # =========================================================
    # TEST 5: Missing results field
    # =========================================================

    print("=" * 80)
    print("TEST 5: Missing results field")
    print("=" * 80)

    state = create_initial_state(
        "Why do customers cancel?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find notes explaining customer "
                "cancellations."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_invalid_vector_search_tool
        ),
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "results"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(
        updated_state["errors"][-1]
    )
    print()

    # =========================================================
    # TEST 6: Invalid individual search result
    # =========================================================

    print("=" * 80)
    print(
        "TEST 6: Invalid individual search result"
    )
    print("=" * 80)

    state = create_initial_state(
        "Why do customers terminate service?"
    )

    state["coordinator_result"] = (
        create_mock_vector_search_coordinator_result(
            task_description=(
                "Find notes explaining why customers "
                "terminate service."
            )
        )
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=(
            mock_invalid_search_result_tool
        ),
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "note"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(
        updated_state["errors"][-1]
    )
    print()

    print("=" * 80)
    print(
        "ALL VECTOR SEARCH AGENT TESTS PASSED"
    )
    print("=" * 80)

In [0]:
def test_vector_search_agent_integration_test() -> None:
    """
    Test the Vector Search Agent with the real
    Databricks Vector Search Tool.
    """

    vector_task = AgentTask(
        task_id="task_1",
        agent_name=VECTOR_SEARCH_AGENT_NAME,
        task_description=(
            "Find customer notes explaining why "
            "customers are likely to cancel service."
        ),
        depends_on=[],
    )

    final_task = AgentTask(
        task_id="task_2",
        agent_name=FINAL_RESPONSE_AGENT_NAME,
        task_description=(
            "Generate the final grounded response."
        ),
        depends_on=[
            VECTOR_SEARCH_AGENT_NAME,
        ],
    )

    coordinator_result = CoordinatorResult(
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        request_type="vector_search",
        reasoning=(
            "The request requires semantic search "
            "over customer notes."
        ),
        execution_plan=[
            vector_task,
            final_task,
        ],
    )

    state = create_initial_state(
        "Why are customers likely to cancel service?"
    )

    state["coordinator_result"] = (
        coordinator_result
    )

    updated_state = run_vector_search_agent(
        state=state,
        vector_search_tool=vector_search_tool,
        num_results=3,
    )

    print("Vector Search Agent Result:")
    print(
        updated_state[
            "agent_results"
        ].get(VECTOR_SEARCH_AGENT_NAME)
    )

    print("\nExecution History:")
    print(
        updated_state["execution_history"]
    )

    print("\nErrors:")
    print(
        updated_state["errors"]
    )

    assert (
        VECTOR_SEARCH_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][VECTOR_SEARCH_AGENT_NAME]

    assert result.status == "success"

    assert result.status == "success"
    assert len(result.results) > 0
    assert len(updated_state["errors"]) == 0
    assert updated_state["execution_history"][-1].status == "success"

    assert result.query is not None

    assert len(result.results) <= 3

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    assert updated_state["errors"] == []

    print(
        "Vector Search Agent integration test passed."
    )


##### 21. Key Learnings

###### The Coordinator decides when the agent runs

- The Vector Search Agent checks: state["coordinator_result"].execution_plan
- It runs only when a task is assigned to: "vector_search_agent"

###### Semantic search uses unstructured text

- The SQL Agent works with structured aggregate data.
- The Vector Search Agent works with customer-note text and retrieves semantically similar records.

###### Dependency injection separates the agent from the implementation

- The agent receives: vector_search_tool as a function argument.
- During testing, this is a mock function. During orchestration, it will be the real Databricks Vector Search function.

###### Empty results are not necessarily errors

- A successful tool response may contain: "results": []
- This means the search worked but found no relevant matches.
- A missing "results" field is a malformed response and should be treated as an error.

###### Tool results are validated before entering shared state

The agent validates:

- Tool status
- Result collection type
- Customer IDs
- Customer notes
- Similarity scores

Only validated results are stored in shared state.

##### 22. Conclusion

The Vector Search Agent now:

- Reads the Coordinator execution plan.
- Finds the assigned Vector Search task.
- Validates task dependencies.
- Resolves the semantic-search query.
- Calls an injected Vector Search Tool.
- Validates and normalizes the tool response.
- Creates a validated VectorSearchAgentResult.
- Stores the result in shared state.
- Handles empty results.
- Handles malformed responses and tool failures.
- Skips cleanly when no Vector Search task is assigned.
- Can be unit tested independently with mock tools.

##### 23. Next Notebook

The next notebook is: 07_retention_agent

The Retention Agent will:

- Read the Coordinator execution plan.
- Determine whether a retention recommendation is required.
- Validate its task dependencies.
- Read Prediction Agent results when required.
- Read Vector Search Agent results when required.
- Call the injected Retention Tool.
- Validate the Retention Tool response.
- Create a validated RetentionAgentResult.
- Store the result in shared state.
- Skip cleanly when no retention task is assigned.
- Record execution history and errors.